In [ ]:
import os
os.environ['KAGGLE_API_TOKEN'] = 'PASTE_YOUR_TOKEN_HERE'
!pip install -U kaggle -q
!kaggle datasets download -d apollo2506/eurosat-dataset
!unzip -q eurosat-dataset.zip -d eurosat_rgb
!ls eurosat_rgb/EuroSAT

Dataset URL: https://www.kaggle.com/datasets/apollo2506/eurosat-dataset
License(s): CC0-1.0
eurosat-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)
replace eurosat_rgb/EuroSAT/AnnualCrop/AnnualCrop_1.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [2]:
import torch, torchvision, numpy as np
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, Subset

data_dir = "eurosat_rgb/EuroSAT"
transform = transforms.Compose([
    transforms.Resize(64),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

full_dataset = datasets.ImageFolder(data_dir, transform=transform)
class_names = full_dataset.classes
print(class_names)
print("Total images:", len(full_dataset))

['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
Total images: 27000


In [3]:
per_class = 300
targets = np.array(full_dataset.targets)
indices = []
for c in range(len(class_names)):
    class_idx = np.where(targets == c)[0]
    indices.extend(np.random.choice(class_idx, per_class, replace=False))
np.random.shuffle(indices)

split = int(0.8 * len(indices))
train_ds = Subset(full_dataset, indices[:split])
val_ds = Subset(full_dataset, indices[split:])
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32)
print("Train:", len(train_ds), "Val:", len(val_ds))

Train: 2400 Val: 600


In [4]:
import torch.nn as nn

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)  # must say cuda, else go set the T4 GPU runtime

model = torchvision.models.resnet18(weights="IMAGENET1K_V1")
model.fc = nn.Linear(model.fc.in_features, len(class_names))
model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

Using device: cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 176MB/s]


In [5]:
epochs = 6
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            preds = model(x).argmax(1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    print(f"Epoch {epoch+1}: loss={total_loss/len(train_loader):.4f}, val_acc={correct/total:.4f}")

Epoch 1: loss=0.8757, val_acc=0.9117
Epoch 2: loss=0.1842, val_acc=0.9317
Epoch 3: loss=0.1033, val_acc=0.9217
Epoch 4: loss=0.0676, val_acc=0.9300
Epoch 5: loss=0.0472, val_acc=0.9233
Epoch 6: loss=0.0427, val_acc=0.9300


In [6]:
torch.save({"model_state": model.state_dict(), "class_names": class_names}, "eurosat_classifier.pt")
from google.colab import files
files.download("eurosat_classifier.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>